In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import gc
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Clean numeric columns
def clean_dataset(df):
    df_clean = df.copy()
    for col in df_clean.select_dtypes(include=[np.number]).columns:
        df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
        if df_clean[col].notnull().any():
            q_low = df_clean[col].quantile(0.001)
            q_high = df_clean[col].quantile(0.999)
            df_clean[col] = df_clean[col].clip(lower=q_low, upper=q_high)
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    return df_clean

# Load and preprocess data
file_path = '../../car_data_v1_3.csv'
df = pd.read_csv(file_path)
df = clean_dataset(df)

# Select columns
selected_columns = ['Cod_fasecolda', 'Modelo', 'Kilometraje', 'Year', 'Month', 'Ubicacion_int', 'Demanda', 'Promedio_estrellas',
                    'Combustible', 'Descripcion_int', 'Gama', 'Blindaje', 'Estado_vehiculo_int', 'Servicio_int', 'Estado_vitrina','Pricing']
df = df[[col for col in selected_columns if col in df.columns]]

# Clean and engineer features
df['Estado_vitrina'] = df['Estado_vitrina'].str.lower().str.strip().str.replace(r'\s+', '_', regex=True)
reemplazos = {'venta_especial_': 'venta_especial', 'venta_especiales': 'venta_especial', 'venta espcial': 'venta_especial'}
df['Estado_vitrina'] = df['Estado_vitrina'].replace(reemplazos)

df['Cod_fasecolda'] = df['Cod_fasecolda'].astype(str).str.zfill(8)
df['Marca_cod'] = df['Cod_fasecolda'].str[:3].astype(int)
df['Tipolo_cod'] = df['Cod_fasecolda'].str[3:5].astype(int)
df['Resto_cod'] = df['Cod_fasecolda'].str[5:].astype(int)

df['Modelo'] = df['Modelo'].replace('-', np.nan)
df = df.dropna(subset=['Modelo'])
df['Kilometraje'] = df['Kilometraje'].replace('-', 0).replace(r'[^\d.-]', '', regex=True)
df['Kilometraje'] = pd.to_numeric(df['Kilometraje'], errors='coerce').fillna(0)
df['Pricing'] = pd.to_numeric(df['Pricing'], errors='coerce')
df = df.dropna(subset=['Pricing'])

# Cast numeric
for i in selected_columns:
    if i not in ['Combustible', 'Gama', 'Estado_vitrina']:
        df[i] = pd.to_numeric(df[i], errors='coerce')

df.drop(columns='Cod_fasecolda', inplace=True)

# Add new features that might improve performance
df['Age'] = 2024 - df['Year']  # Car age
df['KM_per_year'] = df['Kilometraje'] / (df['Age'] + 1)  # Kilometraje per year
df['Gama_Year'] = df['Gama'].astype(str) + '_' + df['Year'].astype(str)  # Interaction feature

# Define categorical and numeric columns
categorical_cols = ['Combustible', 'Gama', 'Estado_vitrina', 'Ubicacion_int', 'Descripcion_int', 
                   'Blindaje', 'Estado_vehiculo_int', 'Servicio_int', 'Gama_Year']
numeric_cols = [col for col in df.columns if col not in categorical_cols + ['Pricing']]

# Label encode categorical
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Target transformation - experiment with different transformations
# Option 1: log1p transform (handles positive values well)
y = np.log1p(df['Pricing'])


# Split for neural network training
X_cat = [df[col].values for col in categorical_cols]
X_num = df[numeric_cols].values

# Scale numerics
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# Split after scaling
X_train_cat, X_test_cat, X_train_num, X_test_num, y_train, y_test = train_test_split(
    list(zip(*X_cat)), X_num_scaled, y, test_size=0.2, random_state=42
)

# Transpose X_cat back to columns
X_train_cat = list(zip(*X_train_cat))
X_test_cat = list(zip(*X_test_cat))

# Prepare inputs for Keras
X_train = [np.array(col) for col in X_train_cat] + [X_train_num]
X_test = [np.array(col) for col in X_test_cat] + [X_test_num]

def build_improved_model():
    inputs = []
    embeddings = []

    # Dynamic embedding dimensions based on cardinality
    for i, col in enumerate(categorical_cols):
        vocab_size = df[col].nunique() + 1
        
        # Better embedding dimension heuristic
        embed_dim = min(50, (vocab_size + 1) // 2)
        
        # Larger embeddings for more important categorical features
        if col in ['Gama', 'Gama_Year']:
            embed_dim = min(100, embed_dim * 2)
            
        input_i = Input(shape=(1,), name=col)
        embed_i = Embedding(input_dim=vocab_size, output_dim=embed_dim)(input_i)
        embed_i = Flatten()(embed_i)
        inputs.append(input_i)
        embeddings.append(embed_i)

    # Numerical input with proper normalization
    input_num = Input(shape=(X_train_num.shape[1],), name='numerical')
    inputs.append(input_num)
    
    # Process numeric inputs
    x_num = BatchNormalization()(input_num)
    x_num = Dense(64, activation='relu')(x_num)
    x_num = BatchNormalization()(x_num)
    
    # Combine categorical and numerical features
    x = Concatenate()(embeddings + [x_num])
    
    # Deeper architecture with residual connections
    x1 = Dense(256, activation='relu')(x)
    x1 = BatchNormalization()(x1)
    x1 = Dropout(0.3)(x1)
    
    x2 = Dense(256, activation='relu')(x1)
    x2 = BatchNormalization()(x2)
    x2 = Dropout(0.3)(x2)
    
    # Residual connection
    x = Concatenate()([x1, x2])
    
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.1)(x)
    
    # Output layer without activation for regression
    out = Dense(1)(x)

    model = Model(inputs=inputs, outputs=out)
    return model

# Custom R2 metric for Keras
class R2Score(tf.keras.metrics.Metric):
    def __init__(self, name='r2_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.y_true_sum = self.add_weight(name='y_true_sum', initializer='zeros')
        self.y_true_squared_sum = self.add_weight(name='y_true_squared_sum', initializer='zeros')
        self.y_pred_sum = self.add_weight(name='y_pred_sum', initializer='zeros')
        self.y_pred_squared_sum = self.add_weight(name='y_pred_squared_sum', initializer='zeros')
        self.y_true_y_pred_sum = self.add_weight(name='y_true_y_pred_sum', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')
        
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        
        self.y_true_sum.assign_add(tf.reduce_sum(y_true))
        self.y_true_squared_sum.assign_add(tf.reduce_sum(tf.square(y_true)))
        self.y_pred_sum.assign_add(tf.reduce_sum(y_pred))
        self.y_pred_squared_sum.assign_add(tf.reduce_sum(tf.square(y_pred)))
        self.y_true_y_pred_sum.assign_add(tf.reduce_sum(y_true * y_pred))
        self.count.assign_add(tf.cast(tf.shape(y_true)[0], tf.float32))
        
    def result(self):
        n = self.count
        numerator = n * self.y_true_y_pred_sum - self.y_true_sum * self.y_pred_sum
        denominator = tf.sqrt((n * self.y_true_squared_sum - tf.square(self.y_true_sum)) * 
                              (n * self.y_pred_squared_sum - tf.square(self.y_pred_sum)))
        
        # Handle edge cases
        coeff = tf.where(tf.equal(denominator, 0.0), 0.0, numerator / denominator)
        return tf.square(coeff)
    
    def reset_state(self):
        self.y_true_sum.assign(0.0)
        self.y_true_squared_sum.assign(0.0)
        self.y_pred_sum.assign(0.0)
        self.y_pred_squared_sum.assign(0.0)
        self.y_true_y_pred_sum.assign(0.0)
        self.count.assign(0.0)



C:\Users\manuel.torres\AppData\Local\Temp\ipykernel_28636\1286644687.py:36: DtypeWarning: Columns (4,11,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


ValueError: Input X contains infinity or a value too large for dtype('float64').

In [ ]:
# Compile and train
model = build_improved_model()

# Use a lower learning rate with warm-up
initial_learning_rate = 0.0005
def lr_schedule(epoch):
    # Learning rate warm-up for first 5 epochs
    if epoch < 5:
        return initial_learning_rate * ((epoch + 1) / 5)
    # Then decay
    return initial_learning_rate * 0.95 ** (epoch - 5)

optimizer = tf.keras.optimizers.Adam(learning_rate=initial_learning_rate)
model.compile(
    optimizer=optimizer, 
    loss='mse',  # Mean Squared Error for regression
    metrics=['mae', R2Score()]  # Track MAE and custom R2
)

# Better callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = LearningRateScheduler(lr_schedule)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train with more epochs but rely on early stopping
history = model.fit(
    X_train, 
    y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=64,  # Smaller batch size for better generalization
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    verbose=1
)

# Evaluate the model
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"Neural Network R2 Score: {r2:.4f}")




In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['r2_score'], label='Train R2')
plt.plot(history.history['val_r2_score'], label='Val R2')
plt.title('Training and Validation R2 Score')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch 

    plt.figure()
    plt.xlabel('Epoch')
    plt.ylabel('Mean Square Error')
    plt.plot(hist['epoch'], hist['mse'], 'r--', label='Trainign Error')
    plt.plot(hist['epoch'], hist['val_mse'], 'b', label='Validation Error')
    plt.ylim([0,0.001])
    plt.legend()
    plt.show()
    
plot_history(history)


In [ ]:
# X_test = np.array(X_test, dtype=np.float32)
# y_test = np.array(y_test, dtype=np.float32)
loss, mae, mse = model.evaluate(X_test, y_test, verbose=0)
print(f'Test MAE: {mae:.2f}, Test MSE: {mse:.2f}')


In [ ]:
y_pred_log = model.predict(X_test, verbose=0)
y_pred = np.expm1(y_pred_log*50)
print(y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))

In [ ]:
print('rmse: ',rmse)
r2 = r2_score(y_test, y_pred_log)
print('R2: ', r2)

In [ ]:
model.save("../my_nn_model_pricing_7.keras")
joblib.dump(scaler, '../scaler_7.pkl')
# el 3 dio mejores resultados